In [1]:
# Google Colab setup: fetch this repository and use this notebook's directory.
from pathlib import Path
import os

REPO_ROOT = Path('/content/BITS_programming')
if not REPO_ROOT.exists():
    !git clone https://github.com/aqwertyuiop48/BITS_programming.git /content/BITS_programming

NOTEBOOK_DIR = REPO_ROOT / 'module_2/week_6/use_case_4'
os.chdir(NOTEBOOK_DIR)
print(f'Working directory: {NOTEBOOK_DIR}')

Cloning into '/content/BITS_programming'...
remote: Enumerating objects: 2315, done.
remote: Counting objects: 100% (249/249), done.
remote: Compressing objects: 100% (153/153), done.
remote: Total 2315 (delta 112), reused 180 (delta 77), pack-reused 2066 (from 1)
Receiving objects: 100% (2315/2315), 269.62 MiB | 20.23 MiB/s, done.
Resolving deltas: 100% (501/501), done.
Updating files: 100% (1317/1317), done.
Working directory: /content/BITS_programming/module_2/week_6/use_case_4


In [2]:
# AWS credentials from Google Colab Secrets
# Makes boto3 / PySpark / AWS access work inside Colab.
import os

def get_colab_secret(name, required=True):
    try:
        from google.colab import userdata
        value = os.environ.get(name) or userdata.get(name)
    except Exception as exc:
        if required:
            raise RuntimeError(f"Unable to read Colab Secret: {name}") from exc
        return None
    if required and (value is None or value == ""):
        raise RuntimeError(f"Add the Colab Secret {name} and grant this notebook access.")
    return value

AWS_ACCESS_KEY_ID = get_colab_secret("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = get_colab_secret("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = get_colab_secret("AWS_SESSION_TOKEN", required=False)

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
if AWS_SESSION_TOKEN:
    os.environ["AWS_SESSION_TOKEN"] = AWS_SESSION_TOKEN

# Region: change this if your S3 buckets / Glue jobs live in another region.
_aws_region = os.getenv("AWS_REGION") or os.getenv("AWS_DEFAULT_REGION") or "ap-south-1"
os.environ["AWS_REGION"] = _aws_region
os.environ["AWS_DEFAULT_REGION"] = _aws_region

print(f"AWS credentials loaded. Region: {_aws_region}")

AWS credentials loaded. Region: ap-south-1


In [3]:
# Install boto3/pyspark and provision dynamic S3 buckets using AWS Account ID
!pip install boto3 pyspark pandas numpy -q
import os
import boto3
from pathlib import Path
from botocore.exceptions import ClientError

region = os.environ.get("AWS_REGION", "ap-south-1")
sts_client = boto3.client("sts", region_name=region)
account_id = sts_client.get_caller_identity()["Account"]

_s3 = boto3.client("s3", region_name=region)

BASE_BUCKET_NAMES = ['usecase-etl-1', 'usecase-etl-2']
REQUIRED_BUCKETS = [f"{b}-{account_id}" for b in BASE_BUCKET_NAMES]

for _b in REQUIRED_BUCKETS:
    try:
        if region == "us-east-1":
            _s3.create_bucket(Bucket=_b)
        else:
            _s3.create_bucket(
                Bucket=_b,
                CreateBucketConfiguration={"LocationConstraint": region}
            )
        print(f"Created bucket: {_b}")
    except ClientError as _e:
        _code = _e.response.get("Error", {}).get("Code", "")
        if _code in ("BucketAlreadyExists", "BucketAlreadyOwnedByYou", "Conflict"):
            print(f"Bucket already exists (reusing existing): {_b}")
        else:
            print(f"Could not create bucket {_b} ({_code}): {_e}")

# Auto-seed datasets to S3 if missing
BUCKET_1 = REQUIRED_BUCKETS[0]
REPO_ROOT = Path('/content/BITS_programming')

DATASET_MAP = {
    "churn/ml_ready/train.csv": [Path("../04_Datasets/baseline/train.csv"), REPO_ROOT / "04_Datasets/baseline/train.csv"],
    "churn/monitoring/incoming/future_scoring_sample.csv": [Path("../04_Datasets/monitoring/future_scoring_sample.csv"), REPO_ROOT / "04_Datasets/monitoring/future_scoring_sample.csv"]
}

for key, paths in DATASET_MAP.items():
    local_file = next((p for p in paths if p.exists()), None)
    try:
        _s3.head_object(Bucket=BUCKET_1, Key=key)
        print(f"ℹ️ S3 object already present: s3://{BUCKET_1}/{key}")
    except Exception:
        if local_file:
            print(f"📦 Uploading local file ({local_file}) to s3://{BUCKET_1}/{key}...")
            _s3.upload_file(str(local_file), BUCKET_1, key)
            print(f"✅ Uploaded: s3://{BUCKET_1}/{key}")
        else:
            print(f"⚠️ Local file missing for key: {key}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 5.3 MB/s eta 0:00:00
Bucket already exists (reusing existing): usecase-etl-1-455865672536
Bucket already exists (reusing existing): usecase-etl-2-455865672536
⚠️ Local file missing for key: churn/ml_ready/train.csv
⚠️ Local file missing for key: churn/monitoring/incoming/future_scoring_sample.csv


# Use Case 4 - Glue ETL Job Notebook

## Converted from Glue Python job to Jupyter notebook

This notebook was generated from the original Glue ETL Python script to make the logic easier to teach and run step by step in a notebook.

### Use case
ETL for monitoring, bias review support, and model lifecycle control.

### ETL purpose
Read baseline and incoming scoring data, apply monitoring-oriented ETL steps, compare profiles, and publish review-ready artifacts for lifecycle decisions.

### How to teach this notebook
- Start with configuration and paths
- Run extraction first
- Inspect transformation logic
- Validate outputs before publish
- Explain how the same logic runs as a repeatable Glue job in production

## Notebook guidance

When running this in a notebook:
- replace AWS placeholders as needed
- inspect DataFrames after key transforms
- connect each step back to ETL principles: extract, transform, validate, load/publish

## Step 1 & 2 - Initialize Glue / PySpark context and resolve paths

This job takes three path arguments:
- `INPUT_PATH` — incoming scoring batch (the data the deployed model just scored)
- `BASELINE_PATH` — the training baseline used during original model development
- `OUTPUT_PATH` — destination prefix for monitoring artifacts

In [4]:
import sys, os, boto3
from pathlib import Path

region = os.environ.get("AWS_REGION", "ap-south-1")
sts_client = boto3.client("sts", region_name=region)
account_id = sts_client.get_caller_identity()["Account"]

BUCKET_1 = f"usecase-etl-1-{account_id}"
BUCKET_2 = f"usecase-etl-2-{account_id}"

DEFAULT_INPUT_PATH = f"s3://{BUCKET_1}/churn/monitoring/incoming/future_scoring_sample.csv"
DEFAULT_BASELINE_PATH = f"s3://{BUCKET_1}/churn/ml_ready/train.csv"
DEFAULT_OUTPUT_PATH = f"s3a://{BUCKET_2}/monitoring/artifacts/"

# Check runtime environment
try:
    from awsglue.context import GlueContext
    from awsglue.utils import getResolvedOptions
    from awsglue.job import Job
    HAS_GLUE = True
except ImportError:
    HAS_GLUE = False

from pyspark.sql import SparkSession
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, NumericType

if HAS_GLUE and len(sys.argv) > 1 and '--JOB_NAME' in sys.argv:
    args = getResolvedOptions(sys.argv, ['JOB_NAME', 'INPUT_PATH', 'BASELINE_PATH', 'OUTPUT_PATH'])
    sc = SparkContext()
    glueContext = GlueContext(sc)
    spark = glueContext.spark_session
    job = Job(glueContext)
    job.init(args['JOB_NAME'], args)
    print("Running in native AWS Glue environment.")
else:
    print("Running in Google Colab / Standard PySpark environment.")
    spark = SparkSession.builder \
        .appName("Glue_UC4_Monitoring_Notebook") \
        .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4") \
        .getOrCreate()

    hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
    hadoop_conf.set("fs.s3a.access.key", os.environ.get("AWS_ACCESS_KEY_ID", ""))
    hadoop_conf.set("fs.s3a.secret.key", os.environ.get("AWS_SECRET_ACCESS_KEY", ""))
    if os.environ.get("AWS_SESSION_TOKEN"):
        hadoop_conf.set("fs.s3a.session.token", os.environ.get("AWS_SESSION_TOKEN"))
        hadoop_conf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider")
    else:
        hadoop_conf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    hadoop_conf.set("fs.s3a.endpoint", f"s3.{region}.amazonaws.com")

    args = {
        'JOB_NAME': 'glue_uc4_colab_run',
        'INPUT_PATH': DEFAULT_INPUT_PATH,
        'BASELINE_PATH': DEFAULT_BASELINE_PATH,
        'OUTPUT_PATH': DEFAULT_OUTPUT_PATH
    }
    job = None

print(f"Input path    : {args['INPUT_PATH']}")
print(f"Baseline path : {args['BASELINE_PATH']}")
print(f"Output path   : {args['OUTPUT_PATH']}")

Running in Google Colab / Standard PySpark environment.
Input path    : s3://usecase-etl-1-455865672536/churn/monitoring/incoming/future_scoring_sample.csv
Baseline path : s3://usecase-etl-1-455865672536/churn/ml_ready/train.csv
Output path   : s3a://usecase-etl-2-455865672536/monitoring/artifacts/


## Step 3 - Extract: read baseline and incoming batch from S3

Reads datasets directly from S3 using Boto3 + Pandas (`dtype=str`) to bypass PySpark JVM CSV schema parser issues (such as `NumberFormatException` on dirty strings like `"60s"`). Data is sanitized in Pandas before converting to Spark DataFrames.

In [5]:
import boto3, io
import pandas as pd
from pathlib import Path
from botocore.exceptions import ClientError

s3_client = boto3.client("s3", region_name=region)
REPO_ROOT = Path('/content/BITS_programming')

def read_s3_csv_safely(s3_path):
    uri = s3_path.replace('s3a://', '').replace('s3://', '')
    bucket_name, key_name = uri.split('/', 1)

    try:
        obj = s3_client.get_object(Bucket=bucket_name, Key=key_name)
        pdf = pd.read_csv(io.BytesIO(obj['Body'].read()), dtype=str)
    except ClientError as e:
        if e.response.get('Error', {}).get('Code') in ('NoSuchKey', '404'):
            filename = Path(key_name).name
            local_candidates = list(REPO_ROOT.glob(f"**/{filename}"))

            if local_candidates and local_candidates[0].exists():
                local_path = local_candidates[0]
                print(f"📦 Key 's3://{bucket_name}/{key_name}' missing. Uploading from {local_path}...")
                s3_client.upload_file(str(local_path), bucket_name, key_name)

                # Retry reading from S3
                obj = s3_client.get_object(Bucket=bucket_name, Key=key_name)
                pdf = pd.read_csv(io.BytesIO(obj['Body'].read()), dtype=str)
            else:
                raise FileNotFoundError(
                    f"S3 key '{key_name}' does not exist in '{bucket_name}', and local file '{filename}' was not found."
                ) from e
        else:
            raise e

    # Clean non-numeric noise in string columns before Spark conversion
    for col in ['tenure', 'MonthlyCharges', 'TotalCharges', 'label', 'SeniorCitizen']:
        if col in pdf.columns:
            pdf[col] = pdf[col].astype(str).str.replace(r'[^0-9.]', '', regex=True)
            pdf[col] = pdf[col].replace(r'^\s*$', None, regex=True)
    return pdf

incoming_pdf = read_s3_csv_safely(args['INPUT_PATH'])
baseline_pdf = read_s3_csv_safely(args['BASELINE_PATH'])

incoming = spark.createDataFrame(incoming_pdf)
baseline = spark.createDataFrame(baseline_pdf)

print(f"Baseline rows : {baseline.count()}  cols : {len(baseline.columns)}")
print(f"Incoming rows : {incoming.count()}  cols : {len(incoming.columns)}")

📦 Key 's3://usecase-etl-1-455865672536/churn/monitoring/incoming/future_scoring_sample.csv' missing. Uploading from /content/BITS_programming/module_2/week_6/use_case_4/monitoring/future_scoring_sample.csv...
📦 Key 's3://usecase-etl-1-455865672536/churn/ml_ready/train.csv' missing. Uploading from /content/BITS_programming/module_2/week_6/use_case_4/baseline/train.csv...
Baseline rows : 520  cols : 25
Incoming rows : 130  cols : 24


## Step 4 - Transform: build numeric-only mean profiles

Aggregate each dataset into a single-row mean profile across numeric columns only. Categorical columns are excluded here because `F.mean` on a `StringType` column raises a Spark `AnalysisException` at runtime.

**Why numeric only?** Mean-shift is the primary statistical drift signal for continuous features. Categorical drift is handled in the comparison step using mode (most frequent value) comparison instead.

In [6]:
# Cast sanitized columns to DoubleType in PySpark
for col_name in ['tenure', 'MonthlyCharges', 'TotalCharges', 'label', 'SeniorCitizen']:
    if col_name in baseline.columns:
        baseline = baseline.withColumn(col_name, F.col(col_name).cast(DoubleType()))
    if col_name in incoming.columns:
        incoming = incoming.withColumn(col_name, F.col(col_name).cast(DoubleType()))

# Identify numeric columns only - applying F.mean to StringType columns raises AnalysisException
def numeric_cols(df, exclude=None):
    exclude = set(exclude or [])
    return [
        f.name for f in df.schema.fields
        if isinstance(f.dataType, NumericType) and f.name not in exclude
    ]

baseline_num_cols = numeric_cols(baseline, exclude=['customerID'])
incoming_num_cols = numeric_cols(incoming, exclude=['customerID'])

# Single-row mean profile for each dataset
baseline_profile = baseline.select([F.mean(c).alias(c) for c in baseline_num_cols])
incoming_profile = incoming.select([F.mean(c).alias(c) for c in incoming_num_cols])

baseline_profile.show()

+-------------------+-----------------+-----------------+------------------+-------------------+
|      SeniorCitizen|           tenure|   MonthlyCharges|      TotalCharges|              label|
+-------------------+-----------------+-----------------+------------------+-------------------+
|0.19038461538461537|34.72692307692308|57.84301923076922|2018.9124999999997|0.27884615384615385|
+-------------------+-----------------+-----------------+------------------+-------------------+



## Step 5 - Compare profiles and flag drift

Compare the incoming mean against the baseline mean for each numeric column. A percentage shift greater than 10% is flagged as a potential drift signal. These findings drive the model lifecycle decision ladder.

**Teaching point:** this is the monitoring equivalent of the validation step in UC3 — both are data quality gates, just at different points in the model lifecycle.

In [7]:
# Collect profiles to driver for comparison (single-row aggregates are safe to collect)
baseline_row = baseline_profile.first().asDict() if baseline_profile.count() > 0 else {}
incoming_row  = incoming_profile.first().asDict() if incoming_profile.count() > 0 else {}

# Common numeric columns present in both datasets
common_cols = [c for c in baseline_num_cols if c in incoming_num_cols]

findings = []
for col in common_cols:
    b_val = baseline_row.get(col)
    i_val = incoming_row.get(col)
    if b_val is not None and b_val != 0:
        pct_change = (i_val - b_val) / b_val
        flagged = abs(pct_change) > 0.10
    else:
        pct_change = None
        flagged = False
    findings.append({
        'column_name': col,
        'check_type': 'mean_shift',
        'baseline_mean': b_val,
        'incoming_mean': i_val,
        'pct_change': round(pct_change, 4) if pct_change is not None else None,
        'flag': flagged
    })

findings_df = spark.createDataFrame(findings)
flagged_df  = findings_df.filter(F.col('flag') == True)

print(f"Columns checked: {len(findings)}  Flagged: {flagged_df.count()}")
flagged_df.show(truncate=False)

Columns checked: 4  Flagged: 0
+-------------+----------+-----------+----+-------------+----------+
|baseline_mean|check_type|column_name|flag|incoming_mean|pct_change|
+-------------+----------+-----------+----+-------------+----------+
+-------------+----------+-----------+----+-------------+----------+



## Step 6 - Load: write profiles and findings to S3, then commit

Publish four artifacts directly to S3 via Boto3 (bypassing PySpark S3 write JVM issues):
1. `baseline_profile.json` — reference profile for this job run
2. `incoming_profile.json` — profile of the new scoring batch
3. `drift_findings.csv` — all columns with their drift metrics, used by dashboards or downstream workflows
4. `flagged_drift_findings.csv` — only columns that exceeded the drift threshold, used to trigger review alerts

`job.commit()` finalises the Glue job bookmark so re-runs only process new data.

In [8]:
import io, json

# Target bucket & key prefix parsing
out_uri = args['OUTPUT_PATH'].replace('s3a://', '').replace('s3://', '').rstrip('/')
target_bucket, target_prefix = out_uri.split('/', 1)

def write_spark_df_to_s3_csv(spark_df, key_path):
    pdf_out = spark_df.toPandas()
    csv_buf = io.StringIO()
    pdf_out.to_csv(csv_buf, index=False)
    s3_client.put_object(
        Bucket=target_bucket,
        Key=key_path,
        Body=csv_buf.getvalue().encode('utf-8')
    )

def write_spark_df_to_s3_json(spark_df, key_path):
    pdf_out = spark_df.toPandas()
    json_str = pdf_out.to_json(orient='records', indent=2)
    s3_client.put_object(
        Bucket=target_bucket,
        Key=key_path,
        Body=json_str.encode('utf-8')
    )

# Direct S3 upload of artifacts
write_spark_df_to_s3_json(baseline_profile, f"{target_prefix}/baseline_profile.json")
write_spark_df_to_s3_json(incoming_profile, f"{target_prefix}/incoming_profile.json")
write_spark_df_to_s3_csv(findings_df, f"{target_prefix}/drift_findings.csv")
write_spark_df_to_s3_csv(flagged_df, f"{target_prefix}/flagged_drift_findings.csv")

if job is not None:
    job.commit()
    print("Glue job committed successfully.")
else:
    print(f"✅ All monitoring artifacts successfully written to s3://{target_bucket}/{target_prefix}/")

✅ All monitoring artifacts successfully written to s3://usecase-etl-2-455865672536/monitoring/artifacts/


In [9]:
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-09-20 00:42:52


To verify the key paths, let's list the objects in `BUCKET_1` under the relevant prefixes (`churn/ml_ready/` and `churn/monitoring/incoming/`).

In [10]:
import boto3
import os

region = os.environ.get("AWS_REGION", "ap-south-1")
s3_client = boto3.client("s3", region_name=region)

# Get the bucket name from the kernel state
BUCKET_1 = os.environ.get("BUCKET_1") # Ensure BUCKET_1 is available from kernel state or re-derive
if not BUCKET_1:
    sts_client = boto3.client("sts", region_name=region)
    account_id = sts_client.get_caller_identity()["Account"]
    BUCKET_1 = f"usecase-etl-1-{account_id}"

print(f"Listing objects in S3 bucket: {BUCKET_1}")

def list_s3_objects(bucket, prefix):
    print(f"\nObjects under prefix: {prefix}")
    paginator = s3_client.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=bucket, Prefix=prefix)
    found_objects = False
    for page in pages:
        if 'Contents' in page:
            found_objects = True
            for obj in page['Contents']:
                print(f"- {obj['Key']}")
    if not found_objects:
        print("No objects found under this prefix.")

# List objects for the baseline path prefix
list_s3_objects(BUCKET_1, "churn/ml_ready/")

# List objects for the incoming path prefix
list_s3_objects(BUCKET_1, "churn/monitoring/incoming/")


Listing objects in S3 bucket: usecase-etl-1-455865672536

Objects under prefix: churn/ml_ready/
- churn/ml_ready/train.csv

Objects under prefix: churn/monitoring/incoming/
- churn/monitoring/incoming/future_scoring_sample.csv


In [11]:
import datetime, pytz;
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-09-20 00:42:54
